# 04. 대전 전체 일별 물량 예측

## 현재 작업 범위

이 노트북은 하이브리드 투스테이지 모델링의 첫 작업 단위로 다음 항목만 수행한다.

1. Train/Test 데이터 로드 및 무결성 재검증
2. Stage 1 평시 모델의 입력 피처와 Target 확정
3. Train 내부 expanding-window 검증 Fold 구성
4. 평시 물량 단순 기준 모델 성능 측정

> 2026년 Test는 구조만 확인하고 모델 선택이나 성능 비교에 사용하지 않는다. ML 모델 학습과 Stage 2 이벤트 효과 모델은 다음 작업 단위에서 진행한다.

## 1. 라이브러리 및 경로 설정

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error


RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "train.csv"
TEST_PATH = PROJECT_ROOT / "data" / "processed" / "test.csv"

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

## 2. Train/Test 로드 및 무결성 검증

In [2]:
EXPECTED_COLUMNS = [
    "접수일자",
    "data_split",
    "접수지역",
    "접수통수",
    "요일",
    "제1기분 자동차세",
    "재산세(건축)",
    "정기분 주민세",
    "주민세(사업소분)",
    "재산세(토지)",
    "제2기분 자동차세",
    "사회보험료 통합",
    "event_count",
    "is_event",
    "month",
    "day",
    "weekday_월",
    "weekday_화",
    "weekday_수",
    "weekday_목",
    "weekday_금",
    "weekday_토",
    "weekday_일",
    "days_since_prev",
    "lag_1",
    "lag_5",
    "rolling_mean_5",
    "rolling_mean_20",
]

assert TRAIN_PATH.exists(), f"Train 파일이 없습니다: {TRAIN_PATH}"
assert TEST_PATH.exists(), f"Test 파일이 없습니다: {TEST_PATH}"

train = pd.read_csv(
    TRAIN_PATH,
    encoding="utf-8-sig",
    parse_dates=["접수일자"],
)
test = pd.read_csv(
    TEST_PATH,
    encoding="utf-8-sig",
    parse_dates=["접수일자"],
)

assert train.columns.tolist() == EXPECTED_COLUMNS
assert test.columns.tolist() == EXPECTED_COLUMNS
assert train.shape == (475, 28)
assert test.shape == (122, 28)
assert train.isna().sum().sum() == 0
assert test.isna().sum().sum() == 0
assert train["접수일자"].is_unique
assert test["접수일자"].is_unique
assert train["접수일자"].is_monotonic_increasing
assert test["접수일자"].is_monotonic_increasing
assert train["data_split"].eq("train").all()
assert test["data_split"].eq("test").all()
assert train["접수일자"].max() < test["접수일자"].min()
assert set(train["접수일자"]).isdisjoint(set(test["접수일자"]))
assert train["is_event"].value_counts().sort_index().to_dict() == {
    0: 334,
    1: 141,
}
assert test["is_event"].value_counts().sort_index().to_dict() == {
    0: 96,
    1: 26,
}

dataset_summary = pd.DataFrame(
    {
        "구분": ["Train", "Test"],
        "행 수": [len(train), len(test)],
        "열 수": [len(train.columns), len(test.columns)],
        "시작일": [
            train["접수일자"].min().date().isoformat(),
            test["접수일자"].min().date().isoformat(),
        ],
        "종료일": [
            train["접수일자"].max().date().isoformat(),
            test["접수일자"].max().date().isoformat(),
        ],
        "평시": [
            int(train["is_event"].eq(0).sum()),
            int(test["is_event"].eq(0).sum()),
        ],
        "이벤트": [
            int(train["is_event"].eq(1).sum()),
            int(test["is_event"].eq(1).sum()),
        ],
        "결측치": [
            int(train.isna().sum().sum()),
            int(test.isna().sum().sum()),
        ],
    }
)

display(dataset_summary)

,구분,행 수,열 수,시작일,종료일,평시,이벤트,결측치
0,Train,475,28,2024-01-30,2025-12-31,334,141,0
1,Test,122,28,2026-01-02,2026-06-30,96,26,0


## 3. Stage 1 Target 및 피처 확정

In [3]:
TARGET_COLUMN = "접수통수"

EVENT_COLUMNS = [
    "제1기분 자동차세",
    "재산세(건축)",
    "정기분 주민세",
    "주민세(사업소분)",
    "재산세(토지)",
    "제2기분 자동차세",
    "사회보험료 통합",
]

STAGE1_FEATURES = [
    "month",
    "day",
    "weekday_월",
    "weekday_화",
    "weekday_수",
    "weekday_목",
    "weekday_금",
    "weekday_토",
    "weekday_일",
    "days_since_prev",
    "lag_1",
    "lag_5",
    "rolling_mean_5",
    "rolling_mean_20",
]

EXCLUDED_MODEL_COLUMNS = [
    "접수일자",
    "data_split",
    "접수지역",
    TARGET_COLUMN,
    "요일",
    "event_count",
    "is_event",
    *EVENT_COLUMNS,
]

assert TARGET_COLUMN in train.columns
assert set(STAGE1_FEATURES).issubset(train.columns)
assert set(EXCLUDED_MODEL_COLUMNS).issubset(train.columns)
assert set(STAGE1_FEATURES).isdisjoint(EXCLUDED_MODEL_COLUMNS)
assert train[STAGE1_FEATURES].notna().all(axis=None)
assert test[STAGE1_FEATURES].notna().all(axis=None)
assert train[TARGET_COLUMN].gt(0).all()
assert test[TARGET_COLUMN].gt(0).all()

baseline_train_all = train.loc[train["is_event"].eq(0)].copy()
event_train_all = train.loc[train["is_event"].eq(1)].copy()

feature_definition = pd.DataFrame(
    {
        "항목": [
            "Target",
            "Stage 1 피처 수",
            "Stage 1 학습 가능 평시 행",
            "Stage 2 후보 이벤트 행",
            "Test 사용 여부(현재 단계)",
        ],
        "값": [
            TARGET_COLUMN,
            len(STAGE1_FEATURES),
            len(baseline_train_all),
            len(event_train_all),
            "구조 검증만 수행",
        ],
    }
)

display(feature_definition)
display(pd.DataFrame({"Stage 1 피처": STAGE1_FEATURES}))

,항목,값
0,Target,접수통수
1,Stage 1 피처 수,14
2,Stage 1 학습 가능 평시 행,334
3,Stage 2 후보 이벤트 행,141
4,Test 사용 여부(현재 단계),구조 검증만 수행


,Stage 1 피처
0,month
1,day
2,weekday_월
3,weekday_화
4,weekday_수
5,weekday_목
6,weekday_금
7,weekday_토
8,weekday_일
9,days_since_prev


## 4. Train 내부 expanding-window Fold 구성

In [4]:
FOLD_SPECS = [
    {
        "fold": "Fold 1",
        "train_start": "2024-01-30",
        "train_end": "2024-06-30",
        "valid_start": "2024-07-01",
        "valid_end": "2024-12-31",
    },
    {
        "fold": "Fold 2",
        "train_start": "2024-01-30",
        "train_end": "2024-12-31",
        "valid_start": "2025-01-01",
        "valid_end": "2025-06-30",
    },
    {
        "fold": "Fold 3",
        "train_start": "2024-01-30",
        "train_end": "2025-06-30",
        "valid_start": "2025-07-01",
        "valid_end": "2025-12-31",
    },
]

time_folds = {}
fold_summary_rows = []

for spec in FOLD_SPECS:
    fold_name = spec["fold"]
    train_start = pd.Timestamp(spec["train_start"])
    train_end = pd.Timestamp(spec["train_end"])
    valid_start = pd.Timestamp(spec["valid_start"])
    valid_end = pd.Timestamp(spec["valid_end"])

    fold_train_all = train.loc[
        train["접수일자"].between(train_start, train_end)
    ].copy()
    fold_valid_all = train.loc[
        train["접수일자"].between(valid_start, valid_end)
    ].copy()

    fold_train_baseline = fold_train_all.loc[
        fold_train_all["is_event"].eq(0)
    ].copy()
    fold_valid_baseline = fold_valid_all.loc[
        fold_valid_all["is_event"].eq(0)
    ].copy()

    assert not fold_train_baseline.empty
    assert not fold_valid_baseline.empty
    assert fold_train_baseline["접수일자"].max() < (
        fold_valid_baseline["접수일자"].min()
    )
    assert fold_train_baseline["접수일자"].max() < test["접수일자"].min()
    assert fold_valid_baseline["접수일자"].max() < test["접수일자"].min()

    time_folds[fold_name] = {
        "train_all": fold_train_all,
        "valid_all": fold_valid_all,
        "train_baseline": fold_train_baseline,
        "valid_baseline": fold_valid_baseline,
    }

    fold_summary_rows.append(
        {
            "Fold": fold_name,
            "학습 전체": len(fold_train_all),
            "학습 평시": len(fold_train_baseline),
            "학습 이벤트": int(fold_train_all["is_event"].eq(1).sum()),
            "검증 전체": len(fold_valid_all),
            "검증 평시": len(fold_valid_baseline),
            "검증 이벤트": int(fold_valid_all["is_event"].eq(1).sum()),
            "학습 종료일": (
                fold_train_all["접수일자"].max().date().isoformat()
            ),
            "검증 시작일": (
                fold_valid_all["접수일자"].min().date().isoformat()
            ),
            "검증 종료일": (
                fold_valid_all["접수일자"].max().date().isoformat()
            ),
        }
    )

fold_summary = pd.DataFrame(fold_summary_rows)
display(fold_summary)

,Fold,학습 전체,학습 평시,학습 이벤트,검증 전체,검증 평시,검증 이벤트,학습 종료일,검증 시작일,검증 종료일
0,Fold 1,104,82,22,124,80,44,2024-06-28,2024-07-01,2024-12-31
1,Fold 2,228,162,66,123,94,29,2024-12-31,2025-01-02,2025-06-30
2,Fold 3,351,256,95,124,78,46,2025-06-30,2025-07-01,2025-12-31


## 5. 회귀 평가 지표

In [5]:
def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    assert y_true.shape == y_pred.shape
    assert np.isfinite(y_true).all()
    assert np.isfinite(y_pred).all()
    assert (y_true > 0).all(), "MAPE 계산을 위해 실제값이 0보다 커야 합니다."

    absolute_error = np.abs(y_true - y_pred)

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE(%)": np.mean(absolute_error / y_true) * 100,
        "WAPE(%)": absolute_error.sum() / np.abs(y_true).sum() * 100,
    }


metric_test = regression_metrics(
    y_true=np.array([100.0, 200.0]),
    y_pred=np.array([90.0, 220.0]),
)

assert np.isclose(metric_test["MAE"], 15.0)
assert np.isclose(metric_test["RMSE"], np.sqrt(250.0))
assert np.isclose(metric_test["MAPE(%)"], 10.0)
assert np.isclose(metric_test["WAPE(%)"], 10.0)

display(pd.DataFrame([metric_test], index=["지표 함수 자체 검증"]))

,MAE,RMSE,MAPE(%),WAPE(%)
지표 함수 자체 검증,15.000,15.811,10.000,10.000


## 6. 평시 단순 기준 모델 평가

다음 기준 모델을 Train 내부 평시 검증 구간에서 비교한다.

- 학습 평시 평균
- 학습 평시 중앙값
- 직전 관측일 물량(`lag_1`)
- 직전 5개 관측일 전 물량(`lag_5`)
- 직전 5개 관측일 평균(`rolling_mean_5`)
- 직전 20개 관측일 평균(`rolling_mean_20`)
- 학습 평시 요일별 평균

Lag·이동평균은 매일 실제 물량이 갱신되는 rolling 1-step 예측 기준이다.

In [6]:
def weekday_mean_prediction(train_frame, valid_frame):
    weekday_means = train_frame.groupby("요일")[TARGET_COLUMN].mean()
    fallback = train_frame[TARGET_COLUMN].median()
    return valid_frame["요일"].map(weekday_means).fillna(fallback).to_numpy()


naive_result_rows = []

for fold_name, fold_data in time_folds.items():
    fold_train = fold_data["train_baseline"]
    fold_valid = fold_data["valid_baseline"]
    y_valid = fold_valid[TARGET_COLUMN].to_numpy()

    predictions = {
        "train_mean": np.full(
            len(fold_valid),
            fold_train[TARGET_COLUMN].mean(),
        ),
        "train_median": np.full(
            len(fold_valid),
            fold_train[TARGET_COLUMN].median(),
        ),
        "lag_1": fold_valid["lag_1"].to_numpy(),
        "lag_5": fold_valid["lag_5"].to_numpy(),
        "rolling_mean_5": fold_valid["rolling_mean_5"].to_numpy(),
        "rolling_mean_20": fold_valid["rolling_mean_20"].to_numpy(),
        "weekday_mean": weekday_mean_prediction(
            fold_train,
            fold_valid,
        ),
    }

    for model_name, y_pred in predictions.items():
        assert len(y_pred) == len(y_valid)
        assert np.isfinite(y_pred).all()

        metrics = regression_metrics(y_valid, y_pred)
        naive_result_rows.append(
            {
                "Fold": fold_name,
                "모델": model_name,
                "검증 평시 행": len(fold_valid),
                **metrics,
            }
        )

naive_fold_results = pd.DataFrame(naive_result_rows)
naive_summary = (
    naive_fold_results.groupby("모델", as_index=False)
    .agg(
        Fold수=("Fold", "nunique"),
        평균_MAE=("MAE", "mean"),
        평균_RMSE=("RMSE", "mean"),
        평균_MAPE=("MAPE(%)", "mean"),
        평균_WAPE=("WAPE(%)", "mean"),
        MAE_표준편차=("MAE", "std"),
    )
    .sort_values(["평균_MAE", "평균_RMSE"])
    .reset_index(drop=True)
)

assert naive_fold_results.shape[0] == len(FOLD_SPECS) * 7
assert naive_summary["Fold수"].eq(3).all()

display(naive_fold_results)
display(naive_summary)

,Fold,모델,검증 평시 행,MAE,RMSE,MAPE(%),WAPE(%)
0,Fold 1,train_mean,80,"35,598.148","42,722.936",70.759,44.676
1,Fold 1,train_median,80,"33,904.900","40,461.359",61.908,42.551
2,Fold 1,lag_1,80,"45,960.875","56,626.282",73.484,57.682
3,Fold 1,lag_5,80,"52,290.238","105,019.598",85.612,65.625
4,Fold 1,rolling_mean_5,80,"43,042.355","55,563.351",78.247,54.019
5,Fold 1,rolling_mean_20,80,"33,499.026","42,210.965",65.593,42.042
6,Fold 1,weekday_mean,80,"29,963.441","38,010.063",54.290,37.605
7,Fold 2,train_mean,94,"42,253.446","64,101.991",72.012,49.781
8,Fold 2,train_median,94,"40,870.319","64,133.556",66.098,48.151
9,Fold 2,lag_1,94,"55,719.777","80,121.113",82.504,65.646


,모델,Fold수,평균_MAE,평균_RMSE,평균_MAPE,평균_WAPE,MAE_표준편차
0,weekday_mean,3,"31,872.351","45,141.790",60.405,41.929,"5,804.907"
1,train_median,3,"34,370.112","45,891.234",65.788,45.115,"6,280.537"
2,rolling_mean_20,3,"36,662.765","48,256.940",70.074,48.039,"8,598.670"
3,train_mean,3,"36,728.515","48,271.412",75.368,48.466,"5,055.433"
4,rolling_mean_5,3,"41,377.300","57,149.321",76.595,54.147,"8,239.767"
5,lag_1,3,"45,859.076","60,492.118",74.253,59.961,"9,911.992"
6,lag_5,3,"49,751.244","92,102.388",90.648,65.798,"4,882.888"


## 7. 첫 작업 단위 결과

In [7]:
best_naive_by_mae = naive_summary.iloc[0]

checkpoint_summary = pd.DataFrame(
    {
        "항목": [
            "Train/Test 무결성",
            "Stage 1 평시 Train",
            "Stage 1 피처 수",
            "Train 내부 Fold",
            "단순 기준 모델 수",
            "평균 MAE 기준 최우수 단순 모델",
            "2026년 Test 성능 확인",
        ],
        "결과": [
            "통과",
            f"{len(baseline_train_all):,}행",
            len(STAGE1_FEATURES),
            len(time_folds),
            naive_summary["모델"].nunique(),
            best_naive_by_mae["모델"],
            "수행하지 않음",
        ],
    }
)

display(checkpoint_summary)

print("첫 모델링 작업 단위 완료")
print("- 데이터 로드 및 무결성 검증: 완료")
print("- Stage 1 피처·Target 정의: 완료")
print("- Train 내부 시간순 Fold: 완료")
print("- 평시 단순 기준 모델 평가: 완료")
print("- ML 모델 및 Stage 2 학습: 수행하지 않음")
print("- 2026년 Test 모델 성능 확인: 수행하지 않음")

,항목,결과
0,Train/Test 무결성,통과
1,Stage 1 평시 Train,334행
2,Stage 1 피처 수,14
3,Train 내부 Fold,3
4,단순 기준 모델 수,7
5,평균 MAE 기준 최우수 단순 모델,weekday_mean
6,2026년 Test 성능 확인,수행하지 않음


첫 모델링 작업 단위 완료
- 데이터 로드 및 무결성 검증: 완료
- Stage 1 피처·Target 정의: 완료
- Train 내부 시간순 Fold: 완료
- 평시 단순 기준 모델 평가: 완료
- ML 모델 및 Stage 2 학습: 수행하지 않음
- 2026년 Test 모델 성능 확인: 수행하지 않음
